In [1]:
# Imports
import os
import json
import concurrent.futures
import re
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
import anthropic

In [2]:
# Client Initialization and helper functions

# Unset proxy env vars before creating the client
for var in ["HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy", "ALL_PROXY", "all_proxy"]:
    os.environ.pop(var, None)

# Explicitly bypass proxy for localhost
os.environ["NO_PROXY"] = "localhost,127.0.0.1"
os.environ["no_proxy"] = "localhost,127.0.0.1"

env_path = ".env"
load_dotenv(dotenv_path=env_path, override=True)

client = anthropic.Anthropic(
    base_url=os.getenv("ANTHROPIC_BASE_URL"),
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)

llm_model = "llama3.2"


def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": llm_model,
        "max_tokens": 1024,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
# Report Builder
def generate_prompt_evaluation_report(evaluation_results):
    total_tests = len(evaluation_results)
    scores = [result["score"] for result in evaluation_results]
    avg_score = mean(scores) if scores else 0
    max_possible_score = 10
    pass_rate = (
        100 * len([s for s in scores if s >= 7]) / total_tests if total_tests else 0
    )

    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prompt Evaluation Report</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                color: #333;
            }}
            .header {{
                background-color: #f0f0f0;
                padding: 20px;
                border-radius: 5px;
                margin-bottom: 20px;
            }}
            .summary-stats {{
                display: flex;
                justify-content: space-between;
                flex-wrap: wrap;
                gap: 10px;
            }}
            .stat-box {{
                background-color: #fff;
                border-radius: 5px;
                padding: 15px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                flex-basis: 30%;
                min-width: 200px;
            }}
            .stat-value {{
                font-size: 24px;
                font-weight: bold;
                margin-top: 5px;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin-top: 20px;
            }}
            th {{
                background-color: #4a4a4a;
                color: white;
                text-align: left;
                padding: 12px;
            }}
            td {{
                padding: 10px;
                border-bottom: 1px solid #ddd;
                vertical-align: top;
            }}
            tr:nth-child(even) {{
                background-color: #f9f9f9;
            }}
            .output-cell {{
                white-space: pre-wrap;
            }}
            .score {{
                font-weight: bold;
                padding: 5px 10px;
                border-radius: 3px;
                display: inline-block;
            }}
            .score-high {{
                background-color: #c8e6c9;
                color: #2e7d32;
            }}
            .score-medium {{
                background-color: #fff9c4;
                color: #f57f17;
            }}
            .score-low {{
                background-color: #ffcdd2;
                color: #c62828;
            }}
            .output {{
                overflow: auto;
                white-space: pre-wrap;
            }}

            .output pre {{
                background-color: #f5f5f5;
                border: 1px solid #ddd;
                border-radius: 4px;
                padding: 10px;
                margin: 0;
                font-family: 'Consolas', 'Monaco', 'Courier New', monospace;
                font-size: 14px;
                line-height: 1.4;
                color: #333;
                box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.1);
                overflow-x: auto;
                white-space: pre-wrap; 
                word-wrap: break-word; 
            }}

            td {{
                width: 20%;
            }}
            .score-col {{
                width: 80px;
            }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box">
                    <div>Total Test Cases</div>
                    <div class="stat-value">{total_tests}</div>
                </div>
                <div class="stat-box">
                    <div>Average Score</div>
                    <div class="stat-value">{avg_score:.1f} / {max_possible_score}</div>
                </div>
                <div class="stat-box">
                    <div>Pass Rate (≥7)</div>
                    <div class="stat-value">{pass_rate:.1f}%</div>
                </div>
            </div>
        </div>

        <table>
            <thead>
                <tr>
                    <th>Scenario</th>
                    <th>Prompt Inputs</th>
                    <th>Solution Criteria</th>
                    <th>Output</th>
                    <th>Score</th>
                    <th>Reasoning</th>
                </tr>
            </thead>
            <tbody>
    """

    for result in evaluation_results:
        prompt_inputs_html = "<br>".join(
            [
                f"<strong>{key}:</strong> {value}"
                for key, value in result["test_case"]["prompt_inputs"].items()
            ]
        )

        criteria_string = "<br>• ".join(result["test_case"]["solution_criteria"])

        score = result["score"]
        if score >= 8:
            score_class = "score-high"
        elif score <= 5:
            score_class = "score-low"
        else:
            score_class = "score-medium"

        html += f"""
            <tr>
                <td>{result["test_case"]["scenario"]}</td>
                <td class="prompt-inputs">{prompt_inputs_html}</td>
                <td class="criteria">• {criteria_string}</td>
                <td class="output"><pre>{result["output"]}</pre></td>
                <td class="score-col"><span class="score {score_class}">{score}</span></td>
                <td class="reasoning">{result["reasoning"]}</td>
            </tr>
        """

    html += """
            </tbody>
        </table>
    </body>
    </html>
    """

    return html

In [4]:
# PromptEvaluator Implementation
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=1):
        self.max_concurrent_tasks = max_concurrent_tasks

    def render(self, template_string, variables):
        placeholders = re.findall(r"{([^{}]+)}", template_string)

        result = template_string
        for placeholder in placeholders:
            if placeholder in variables:
                result = result.replace(
                    "{" + placeholder + "}", str(variables[placeholder])
                )

        return result.replace("{{", "{").replace("}}", "}")

    def generate_unique_ideas(self, task_description, prompt_inputs_spec, num_cases):
        """Generate a list of unique ideas for test cases based on the task description"""

        prompt = """
        Generate {num_cases} unique, diverse ideas for testing a prompt that accomplishes this task:
        
        <task_description>
        {task_description}
        </task_description>

        The prompt will receive the following inputs
        <prompt_inputs>
        {prompt_inputs_spec}
        </prompt_inputs>
        
        Each idea should represent a distinct scenario or example that tests different aspects of the task.
        
        Output Format:
        Provide your response as a structured JSON array where each item is a brief description of the idea.
        
        Example:
        ```json
        [
            "Testing with technical computer science terminology",
            "Testing with medical research findings",
            "Testing with complex mathematical concepts",
            ...
        ]
        ```
        
        Ensure each idea is:
        - Clearly distinct from the others
        - Relevant to the task description
        - Specific enough to guide generation of a full test case
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output

        Remember, only generate {num_cases} unique ideas
        """

        system_prompt = "You are a test scenario designer specialized in creating diverse, unique testing scenarios."

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": str # {val},'

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "task_description": task_description,
                "num_cases": num_cases,
                "prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=1.0,
        )

        return json.loads(text)

    def generate_test_case(self, task_description, idea, prompt_inputs_spec={}):
        """Generate a single test case based on the task description and a specific idea"""

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": "EXAMPLE_VALUE", // {val}\n'

        allowed_keys = ", ".join([f'"{key}"' for key in prompt_inputs_spec.keys()])

        prompt = """
        Generate a single detailed test case for a prompt evaluation based on:
        
        <task_description>
        {task_description}
        </task_description>
        
        <specific_idea>
        {idea}
        </specific_idea>
        
        <allowed_input_keys>
        {allowed_keys}
        </allowed_input_keys>
        
        Output Format:
        ```json
        {{
            "prompt_inputs": {{
            {example_prompt_inputs}
            }},
            "solution_criteria": ["criterion 1", "criterion 2", ...] // Concise list of criteria for evaluating the solution, 1 to 4 items
        }}
        ```
        
        IMPORTANT REQUIREMENTS:
        - You MUST ONLY use these exact input keys in your prompt_inputs: {allowed_keys}        
        - Do NOT add any additional keys to prompt_inputs
        - All keys listed in allowed_input_keys must be included in your response
        - Make the test case realistic and practically useful
        - Include measurable, concise solution criteria
        - The solution criteria should ONLY address the direct requirements of the task description and the generated prompt_inputs
        - Avoid over-specifying criteria with requirements that go beyond the core task
        - Keep solution criteria simple, focused, and directly tied to the fundamental task
        - The test case should be tailored to the specific idea provided
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output
        - DO NOT include any fields beyond those specified in the output format

        Here's an example of a sample input with an ideal output:
        <sample_input>
        <sample_task_description>
        Extract topics out of a passage of text
        </sample_task_description>
        <sample_specific_idea>
        Testing with a text that contains multiple nested topics and subtopics (e.g., a passage about renewable energy that covers solar power economics, wind turbine technology, and policy implications simultaneously)
        </sample_specific_idea>

        <sample_allowed_input_keys>
        "content"
        </sample_allowed_input_keys>
        </sample_input>
        <ideal_output>
        ```json
        {
            "prompt_inputs": {
                "content": "The transition to renewable energy encompasses numerous interdependent dimensions. Solar photovoltaic technology has seen dramatic cost reductions, with panel efficiency improving 24% since 2010 while manufacturing costs declined by 89%, making it economically competitive with fossil fuels in many markets. Concurrently, wind energy has evolved through innovative turbine designs featuring carbon-fiber composite blades and advanced control systems that increase energy capture by 35% in low-wind conditions."
            },
            "solution_criteria": [
                "Includes all topics mentioned"   
            ]
        }
        ```
        </ideal_output>
        This is ideal output because the solution criteria is concise and doesn't ask for anything outside of the scope of the task description.
        """

        system_prompt = "You are a test case creator specializing in designing evaluation scenarios."

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "allowed_keys": allowed_keys,
                "task_description": task_description,
                "idea": idea,
                "example_prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=0.7,
        )

        test_case = json.loads(text)
        test_case["task_description"] = task_description
        test_case["scenario"] = idea

        return test_case

    def generate_dataset(
        self,
        task_description,
        prompt_inputs_spec={},
        num_cases=1,
        output_file="dataset.json",
    ):
        """Generate test dataset based on task description and save to file"""
        ideas = self.generate_unique_ideas(
            task_description, prompt_inputs_spec, num_cases
        )

        dataset = []
        completed = 0
        total = len(ideas)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_idea = {
                executor.submit(
                    self.generate_test_case,
                    task_description,
                    idea,
                    prompt_inputs_spec,
                ): idea
                for idea in ideas
            }

            for future in concurrent.futures.as_completed(future_to_idea):
                try:
                    result = future.result()
                    completed += 1
                    current_percentage = int((completed / total) * 100)
                    milestone_percentage = (current_percentage // 20) * 20

                    if milestone_percentage > last_reported_percentage:
                        print(f"Generated {completed}/{total} test cases")
                        last_reported_percentage = milestone_percentage

                    dataset.append(result)
                except Exception as e:
                    print(f"Error generating test case: {e}")

        with open(output_file, "w") as f:
            json.dump(dataset, f, indent=2)

        return dataset

    def grade_output(self, test_case, output, extra_criteria):
        """Grade the output of a test case using the model"""

        prompt_inputs = ""
        for key, value in test_case["prompt_inputs"].items():
            val = value.replace("\n", "\\n")
            prompt_inputs += f'"{key}":"{val}",\n'

        extra_criteria_section = ""
        if extra_criteria:
            extra_criteria_template = """
            Mandatory Requirements - ANY VIOLATION MEANS AUTOMATIC FAILURE (score of 3 or lower):
            <extra_important_criteria>
            {extra_criteria}
            </extra_important_criteria>
            """
            extra_criteria_section = self.render(
                dedent(extra_criteria_template),
                {"extra_criteria": extra_criteria},
            )

        eval_template = """
        Your task is to evaluate the following AI-generated solution with EXTREME RIGOR.

        Original task description:
        <task_description>
        {task_description}
        </task_description>

        Original task inputs:
        <task_inputs>
        {{ {prompt_inputs} }}
        </task_inputs>

        Solution to Evaluate:
        <solution>
        {output}
        </solution>

        Criteria you should use to evaluate the solution:
        <criteria>
        {solution_criteria}
        </criteria>

        {extra_criteria_section}

        Scoring Guidelines:
        * Score 1-3: Solution fails to meet one or more MANDATORY requirements
        * Score 4-6: Solution meets all mandatory requirements but has significant deficiencies in secondary criteria
        * Score 7-8: Solution meets all mandatory requirements and most secondary criteria, with minor issues
        * Score 9-10: Solution meets all mandatory and secondary criteria

        IMPORTANT SCORING INSTRUCTIONS:
        * Grade the output based ONLY on the listed criteria. Do not add your own extra requirements.
        * If a solution meets all of the mandatory and secondary criteria give it a 10
        * Don't complain that the solution "only" meets the mandatory and secondary criteria. Solutions shouldn't go above and beyond - they should meet the exact listed criteria.
        * ANY violation of a mandatory requirement MUST result in a score of 3 or lower
        * The full 1-10 scale should be utilized - don't hesitate to give low scores when warranted

        Output Format
        Provide your evaluation as a structured JSON object with the following fields, in this specific order:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement
        - "reasoning": A concise explanation of your overall assessment
        - "score": A number between 1-10

        Respond with JSON. Keep your response concise and direct.
        Example response shape:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
        """

        eval_prompt = self.render(
            dedent(eval_template),
            {
                "task_description": test_case["task_description"],
                "prompt_inputs": prompt_inputs,
                "output": output,
                "solution_criteria": "\n".join(test_case["solution_criteria"]),
                "extra_criteria_section": extra_criteria_section,
            },
        )

        messages = []
        add_user_message(messages, eval_prompt)
        add_assistant_message(messages, "```json")
        eval_text = chat(
            messages,
            stop_sequences=["```"],
            temperature=0.0,
        )
        return json.loads(eval_text)

    def run_test_case(self, test_case, run_prompt_function, extra_criteria=None):
        """Run a test case and grade the result"""
        output = run_prompt_function(test_case["prompt_inputs"])

        model_grade = self.grade_output(test_case, output, extra_criteria)
        model_score = model_grade["score"]
        reasoning = model_grade["reasoning"]

        return {
            "output": output,
            "test_case": test_case,
            "score": model_score,
            "reasoning": reasoning,
        }

    def run_evaluation(
        self,
        run_prompt_function,
        dataset_file,
        extra_criteria=None,
        json_output_file="output.json",
        html_output_file="output.html",
    ):
        """Run evaluation on all test cases in the dataset"""
        with open(dataset_file, "r") as f:
            dataset = json.load(f)

        results = []
        completed = 0
        total = len(dataset)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_test_case = {
                executor.submit(
                    self.run_test_case,
                    test_case,
                    run_prompt_function,
                    extra_criteria,
                ): test_case
                for test_case in dataset
            }

            for future in concurrent.futures.as_completed(future_to_test_case):
                result = future.result()
                completed += 1
                current_percentage = int((completed / total) * 100)
                milestone_percentage = (current_percentage // 20) * 20

                if milestone_percentage > last_reported_percentage:
                    print(f"Graded {completed}/{total} test cases")
                    last_reported_percentage = milestone_percentage
                results.append(result)

        average_score = mean([result["score"] for result in results])
        print(f"Average score: {average_score}")

        with open(json_output_file, "w") as f:
            json.dump(results, f, indent=2)

        html = generate_prompt_evaluation_report(results)
        with open(html_output_file, "w", encoding="utf-8") as f:
            f.write(html)

        return results

In [ ]:
# Create an instance of PromptEvaluator
# Increase `max_concurrent_tasks` for greater concurrency, but beware of rate limit errors!
evaluator = PromptEvaluator(max_concurrent_tasks=2)

In [7]:
dataset = evaluator.generate_dataset(
    # Describe the purpose or goal of the prompt you're trying to test
    task_description="Write a compact, concise 1 day meal plan for a single athlete. Avoid using complicated and special characters to avoid json parsing errors",
    # Describe the different inputs that your prompt requires
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of tthe athlete",
        "restrictions": "Dietary restrictions of the athlete"
    },
    # Where to write the generated dataset
    output_file="dataset.json",
    # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
    num_cases=10,
)

Generated 2/10 test cases
Generated 4/10 test cases
Generated 6/10 test cases
Generated 8/10 test cases
Generated 10/10 test cases


In [16]:

    # prompt = f"""
    # Generate a one-day meal plan for as athlete that meets their dietary restrictions.

    # - Height: {prompt_inputs["height"]}
    # - Weight: {prompt_inputs["weight"]}
    # - Goal: {prompt_inputs["goal"]}
    # - Dietary restrictions: {prompt_inputs["restrictions"]}

    # Guidelines:
    # 1. Include accurate daily calorie amount
    # 2. Show protein, fat and carb amounts
    # 3. Specify when to eattt each meal
    # 4. Use only foods that fit resttricttions
    # 5. List all portion sizes in grams
    # 6. Keep budget-friendly if mentioned
    # """
    # prompt = f"""
    # Generate a one-day meal plan for as athlete that meets their dietary restrictions.

    # <athlete_information>
    # - Height: {prompt_inputs["height"]}
    # - Weight: {prompt_inputs["weight"]}
    # - Goal: {prompt_inputs["goal"]}
    # - Dietary restrictions: {prompt_inputs["restrictions"]}
    # </athlete_information>
    
    # Follow these steps:
    # 1. Calculate daily calories needed
    # 2. Figure out protein, fat and carb amount
    # 3. Plan meal timing around workouts
    # 4. Choose foods that fit restrictions
    # 5. Settt portion sizes in grams
    # 6. Adjust budgett if needed
    # """

# <sample_input>
#     height: 175
#     weight: 65
#     goal: lose weight
#     restrictions: vegetarian
#     </sample_input>

#     <ideal_output>
#     Based on the provided information, I've created a one-day meal plan for the vegetarian athlete looking to lose weight.

#     **Athlete Information:**

#     * Height: 175 cm (5'9")
#     * Weight: 65 kg (143 lbs)
#     * Goal: Lose weight
#     * Dietary restrictions: Vegetarian

#     **Daily Calorie Intake:**
#     To support weight loss, the daily calorie intake for this athlete is approximately 1800 calories.

#     **Macronutrient Breakdown:**

#     * Protein: 1.6 grams/kg body weight = 104g ( approx. 20% of total calories)
#     * Fat: 0.5g/kg body weight = 32g (approx. 15% of total calories)
#     * Carbohydrates: 2-3 grams/kg body weight = 130g (approx. 60% of total calories)

#     **Meal Plan:**

#     **Breakfast (6:00 am)**

#     * Oatmeal with almond milk, banana, and walnuts
#         + 250g oatmeal cooked with 150ml water
#         + 120g almond milk
#         + 100g sliced banana
#         + 20g chopped walnuts
#         + 10g honey (optional)

#     **Mid-Morning Snack (10:00 am)**

#     * Carrot sticks with hummus (60g)
#         + 100g carrot sticks
#         + 50g hummus

#     **Lunch (12:30 pm)**

#     * Whole-grain wrap with avocado, lettuce, tomato, and chickpeas
#         + 150g whole-grain wrap
#         + 40g mashed avocado
#         + 20g chopped lettuce
#         + 25g sliced tomato
#         + 100g cooked chickpeas
#         + 10g low-fat yogurt

#     **Mid-Afternoon Snack (3:30 pm)**

#     * Apple slices with almond butter (80g)
#         + 150g apple slices
#         + 30g almond butter

#     **Dinner (6:30 pm)**

#     * Quinoa and vegetable stir-fry with tofu
#         + 150g cooked quinoa
#         + 100g mixed vegetables (bell peppers, broccoli, carrots) cooked in 20g coconut oil
#         + 50g cubed firm tofu
#         + 10g soy sauce (optional)

#     **Before Bed Snack (9:00 pm)**

#     * Greek yogurt with berries and chia seeds (60g)
#         + 100g Greek yogurt
#         + 30g mixed berries
#         + 10g chia seeds

#     **Total Daily Intake:**

#     * Calories: 1785
#     * Protein: 104g
#     * Fat: 32g
#     * Carbohydrates: 130g

#     This meal plan provides a balanced mix of protein, healthy fats, and complex carbohydrates while meeting the athlete's dietary restrictions. The portion sizes are based on the athlete's individual needs, and the calorie intake is carefully calculated to support weight loss.
#     </ideal_output>
#     This example meal plan was rated high. While the solution meets most of the mandatory requirements, it falls short in providing a calorie intake that supports weight loss. However, it does provide a balanced mix of protein, healthy fats, and complex carbohydrates.

# Define and run the prompt you want to evaluate, returning the raw model output
# This function is executed once for each test case
def run_prompt(prompt_inputs):

    xml_file = r"2026/claude_certified_architect/examples.xml"

    prompt = f"""
    Generate a one-day meal plan for as athlete that meets their dietary restrictions.

    <athlete_information>
    - Height: {prompt_inputs["height"]}
    - Weight: {prompt_inputs["weight"]}
    - Goal: {prompt_inputs["goal"]}
    - Dietary restrictions: {prompt_inputs["restrictions"]}
    </athlete_information>

    Guidelines:
    1. Include accurate daily calorie amount
    2. Show protein, fat and carb amounts
    3. Specify when to eattt each meal
    4. Use only foods that fit resttricttions
    5. List all portion sizes in grams
    6. Keep budget-friendly if mentioned
    
    Here is a collection of sample inputs outputs score and reasoning in the xml file:
    {xml_file}
    
    """
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [ ]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact food, portions and timing
    """
)

# Iterattion #2: Better prompt content/text/explanation
# Average score: 6.875

# Iteration #3: 
# Average score: 7.111111111111111

# Iteration #4: prompt with Guidelines
# Average score: 6.9

# Iteration #5: prompt with Steps to follow
# Average score: 7

# Iteration #6: prompt with xml tagged data
# Average score: 6.8

# Iteration #7: one example provided
# Average score: 7.4

# Iteration #8: multiple examples via xml filehi 
# Average score: 7.1

Graded 2/10 test cases
Graded 4/10 test cases
Graded 6/10 test cases
Graded 8/10 test cases
Graded 10/10 test cases
Average score: 7.1


# Learning Exercise

In [19]:
dataset = evaluator.generate_dataset(
    # Describe the purpose or goal of the prompt you're trying to test
    task_description="Extract topics out of a passage of text from a scholarly artticle into a JSON array of strings",
    # Describe the different inputs that your prompt requires
    prompt_inputs_spec={
        "content": "One paragraph of text from a scholarly journal writtten in English "
    },
    # Where to write the generated dataset
    output_file="dataset.json",
    # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
    num_cases=10,
)

Generated 2/10 test cases
Generated 4/10 test cases
Generated 6/10 test cases
Generated 8/10 test cases
Generated 10/10 test cases


In [38]:
# bad prompt
    # prompt = f"""
    # What topics are in here?
    
    # {prompt_inputs["content"]}
    # """

    # prompt = f"""
    # Extract key ttopics mentioned in a passage of text from a scholarly journal into a JSON array of strings.
    # {prompt_inputs["content"]}
    # """

#    Example of input and corresponding output
#     <sample input>
#     The concept of circular economy has been gaining traction in recent years, driven by growing concerns about environmental sustainability and resource depletion. A circular economy is based on the principles of reuse, recycling, and waste reduction, aiming to minimize waste and maximize resource efficiency. By adopting a circular economy approach, industries can reduce their environmental footprint, improve supply chain resilience, and enhance competitiveness. For instance, companies like Patagonia and H&M have implemented garment collecting initiatives, encouraging customers to return used clothing in exchange for discounts on new purchases. Additionally, the development of biodegradable materials and advanced recycling technologies has created new opportunities for sustainable product design and manufacturing. However, despite these progress, challenges persist, including the need for standardized measurement and certification schemes, as well as effective policies and regulations to support circular economy practices.
#     </sample input>
#     <ideal output>
#     "circular economy", "environmental sustainability", "resource depletion", "reuse", "recycling", "waste reduction", "resource efficiency", "supply chain resilience", "competitiveness", "biodegradable materials", "advanced recycling technologies", "sustainable product design", "manufacturing", "standardized measurement and certification schemes", "effective policies and regulations"
#     </ideal output>

def run_prompt(prompt_inputs):
    prompt = f"""
    Extract key topics mentioned from a passage of text from a scholarly journal into a JSON list of strings as the actual output.
    Do not generate code to get the JSON or any other instructions but jus the simple JSON output

   Example of input and corresponding JSON output shared as a list of strings 
    <sample input>
    The concept of circular economy has been gaining traction in recent years, driven by growing concerns about environmental sustainability and resource depletion. A circular economy is based on the principles of reuse, recycling, and waste reduction, aiming to minimize waste and maximize resource efficiency. By adopting a circular economy approach, industries can reduce their environmental footprint, improve supply chain resilience, and enhance competitiveness. For instance, companies like Patagonia and H&M have implemented garment collecting initiatives, encouraging customers to return used clothing in exchange for discounts on new purchases. Additionally, the development of biodegradable materials and advanced recycling technologies has created new opportunities for sustainable product design and manufacturing. However, despite these progress, challenges persist, including the need for standardized measurement and certification schemes, as well as effective policies and regulations to support circular economy practices.
    </sample input>
    <ideal output>
    "circular economy", "environmental sustainability", "resource depletion", "reuse", "recycling", "waste reduction", "resource efficiency", "supply chain resilience", "competitiveness", "biodegradable materials", "advanced recycling technologies", "sustainable product design", "manufacturing", "standardized measurement and certification schemes", "effective policies and regulations"
    </ideal output>
    
    <text>
    {prompt_inputs["content"]}
    </text>

    * Make sure that output only contains a JSON
    * Make sure that there is no additional commentary or verbiage in the output
    * Make sure that tthe key solution criteria are strictly met    
    """
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [39]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
    - Contains a JSON array of strings, containing each topic mentioned in the article.
    - The strings should contain only a topic without any extra commentary
    - Response should contain the JSON array and nothing else
    """
)

# Iteration #1: bad prompt
# Average score: 5.5

# Iteration #2: slightly improved
# Average score: 6.3

# Iteration #3: improved prompt - no impact, same score
# Average score: 6.3

# Iteration #3: 1-shot example, xml tags - made it worse
# Average score: 5.8

Graded 2/10 test cases
Graded 4/10 test cases
Graded 6/10 test cases
Graded 8/10 test cases
Graded 10/10 test cases
Average score: 6.2
